# Does `pad_stride` explain the MGLPDSNet-vs-LPDSNet gap at init?**Hypothesis.** The two models disagree at initialisation not because of the multigridalgorithm, but because they pad the k-space operator onto *different* grids.`MGLPDSNet.pad_stride = s * 2**(levels - 1)`:| config | `K` | levels | `pad_stride` ||---|---|---|---|| LPDSNet | `30` | 1 | **2** || MGLPDSNet | `[6, [4, 4, 6]]` | 3 | **8** |Under `preproc="kspace"`, `kspace_pre_process` pads the **operator** up to that stride:the sampling mask is nearest-neighbour resampled and the coil maps are reflect-padded.`preprocessing/kspace.py` says what that costs:> This is inherently approximate -- a Fourier transform on a larger grid is a different> transform, so the padded operator is not the original one -- and it is why the configs> size their data to a multiple of `pad_stride`, where the pad is empty and this is the> identity.An even-sized image is always a multiple of 2, so **LPDSNet never pads**. MGLPDSNet padswhenever the size is not a multiple of 8 -- silently, with no error.**Prediction.** Sweep the image size. LPDSNet should be flat. MGLPDSNet should be *equal orbetter* when `size % 8 == 0` and collapse otherwise, periodically in 8. If instead bothdegrade together, or MGLPDSNet is uniformly worse at every size, the hypothesis is wrongand the problem really is the V-cycle.Everything below is at **initialisation** -- no training, no checkpoints.

In [ ]:
import os, sys, math, jsonimport numpy as npimport torchimport matplotlib.pyplot as plt# repo root: the notebook is expected to live in ImMAP/notebooks/ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))if not os.path.isfile(os.path.join(ROOT, "train.py")):    ROOT = os.path.abspath(os.getcwd())sys.path.insert(0, ROOT)print("repo:", ROOT)from models.mg_lpds import MGLPDSNetfrom operators import FFT2D, Mask, Sensefrom operators.noise import mri_awgnfrom physics.mask import make_acc_maskfrom preprocessing.kspace import kspace_pre_processDEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")SEED = 0R = 8                 # acceleration, matches the R8 configsACS = 20SIGMA = 0.005         # FIXED, not drawn -- otherwise size-to-size differences                      # would partly be noise draws rather than the effectprint("device:", DEVICE)

## 1. Ground truthPrefer a real preprocessed fastMRI slice (coil-combined image + its sensitivity maps).Falls back to an analytic phantom with synthetic smooth maps so the notebook runs anywhere.The conclusion does not depend on which one you get -- the effect is a property of theoperator, not the image -- but a real slice makes the picture panel readable.

In [ ]:
# Point this at a preprocessed volume; anything with `image` and `smaps` datasets works.SMAP_ROOT = "../datasets/fastmri_preprocessed/brain_T2W_coil_combined/train"SCALE_FAC = 2e3       # brain, from datasets/fastmri/loader.py::FASTMRI_PATHSSLICE = 4def load_real(smap_root, sl=SLICE, scale_fac=SCALE_FAC):    import h5py, glob    root = smap_root if os.path.isabs(smap_root) else os.path.join(ROOT, smap_root)    files = sorted(glob.glob(os.path.join(root, "*.h5")))    if not files:        raise FileNotFoundError(root)    with h5py.File(files[0], "r") as f:        image = np.asarray(f["image"][sl:sl + 1])        smaps = np.asarray(f["smaps"][sl:sl + 1])    gt = torch.from_numpy(image).to(torch.complex64).reshape(1, 1, *image.shape[-2:])    sm = torch.from_numpy(smaps).to(torch.complex64)    sm = sm.reshape(1, -1, *sm.shape[-2:])    print(f"real slice from {os.path.basename(files[0])}  image {tuple(gt.shape)}  "          f"coils {sm.shape[1]}")    return gt * scale_fac, smdef make_phantom(H=320, W=320, NC=8, seed=SEED):    g = torch.Generator().manual_seed(seed)    yy, xx = torch.meshgrid(torch.linspace(-1, 1, H), torch.linspace(-1, 1, W),                            indexing="ij")    mag = (0.25           + 0.9 * torch.exp(-(xx ** 2 + yy ** 2) * 2.5)           + 0.5 * (((xx - .30) ** 2 + (yy + .20) ** 2) < .02).float()           + 0.4 * (((xx + .35) ** 2 + (yy - .30) ** 2) < .012).float()           - 0.3 * (((xx + .05) ** 2 + (yy + .40) ** 2) < .008).float())    gt = (mag * torch.exp(1j * 1.5 * xx * yy)).to(torch.complex64)[None, None]    # smooth complex coil maps: low-order polynomials, then unit-RSS normalised    cx = torch.cos(torch.arange(NC) * 2 * math.pi / NC)    cy = torch.sin(torch.arange(NC) * 2 * math.pi / NC)    sm = torch.stack([torch.exp(-((xx - a) ** 2 + (yy - b) ** 2) * 0.8)                      * torch.exp(1j * (a * xx + b * yy) * 2.0)                      for a, b in zip(cx, cy)]).to(torch.complex64)[None]    sm = sm / (sm.abs().pow(2).sum(1, keepdim=True).sqrt() + 1e-8)    print(f"analytic phantom  image (1, 1, {H}, {W})  coils {NC}")    return gt, smtry:    GT_FULL, SMAPS_FULL = load_real(SMAP_ROOT)    SOURCE = "fastMRI"except Exception as e:    print(f"[falling back to phantom: {type(e).__name__}: {e}]")    GT_FULL, SMAPS_FULL = make_phantom()    SOURCE = "phantom"# unit-RSS check: mri_awgn's sigma only means "noise std of the coil-combined# adjoint" when this holds, which is what makes sigma comparable across sizes.rss = SMAPS_FULL.abs().pow(2).sum(1).sqrt()print(f"smaps RSS: min {rss.min():.4f}  max {rss.max():.4f}  "      f"(1.0 expected where there is coil support)")

## 2. One problem per crop sizeCrop **in the image domain**, then simulate k-space from the cropped image. The forwardoperator is rebuilt at the cropped size, so every size is a self-consistent, correctlyposed problem. The *only* thing that varies across the sweep is whether the size happensto be divisible by 8.Sigma and the mask pattern are fixed, so any difference between sizes is the operator,not the draw.

In [ ]:
def center_crop(t, size):    H, W = t.shape[-2:]    h, w = size    top, left = (H - h) // 2, (W - w) // 2    return t[..., top:top + h, left:left + w]def make_problem(size, gt_full=None, smaps_full=None, sigma=SIGMA, R=R, acs=ACS,                 seed=SEED, device=DEVICE):    # Crop to `size`, rebuild E at that size, simulate y. Returns everything the    # networks need plus the ground truth to score against.    gt_full = GT_FULL if gt_full is None else gt_full    smaps_full = SMAPS_FULL if smaps_full is None else smaps_full    hw = (size, size) if isinstance(size, int) else tuple(size)    gt = center_crop(gt_full, hw).to(device)    sm = center_crop(smaps_full, hw).to(device)    mask = make_acc_mask(hw, R, acs_lines=acs, device=device)    while mask.dim() < 4:        mask = mask.unsqueeze(0)    E = Mask(mask) @ FFT2D() @ Sense(sm)    torch.manual_seed(seed)                      # same noise realisation every size    y, _, _ = mri_awgn(gt, mask, sm, sigma, "uniform")    sig = torch.full((1, 1, 1, 1), float(sigma), device=device)    return dict(gt=gt, E=E, y=y, sigma=sig, size=hw)def psnr(est, ref):    mse = float(((est.abs() - ref.abs()) ** 2).mean())    return 10 * math.log10(float(ref.abs().max()) ** 2 / max(mse, 1e-20))def nrmse(est, ref):    return float((est - ref).abs().norm() / ref.abs().norm())

## 3. The two networks, at initialisationBuilt **once** and reused at every size -- the parameters do not depend on the inputshape, and building is the expensive part (`spectral_normalize` runs a power method perlayer). Same seed for both, so their level-0 dictionaries start from the same draw.Model params are read from your generated configs so this tracks the real experiment.

In [ ]:
CFG_DIR = os.path.join(ROOT, "config", "brain", "mg")def params_from(tag):    with open(os.path.join(CFG_DIR, f"{tag}_R8.json")) as f:        return json.load(f)["model"]["params"]try:    P_FLAT = params_from("lpdsnet")    P_MG = params_from("mglpds")    print("model params from config/brain/mg/*_R8.json")except FileNotFoundError:    BASE = dict(M=169, C=1, P=7, s=2, widen=1, degrees=1, lam0=1e-3, tau0=0.5,                theta0=0.0, alpha0=1.0, is_complex=True, preproc="kspace",                resize_noise=True)    P_FLAT, P_MG = dict(BASE, K=30), dict(BASE, K=[6, [4, 4, 6]])    print("configs not found; using the generator's defaults")print("  LPDSNet  ", {k: P_FLAT[k] for k in ("K", "M", "P", "s", "preproc")})print("  MGLPDSNet", {k: P_MG[k] for k in ("K", "M", "P", "s", "preproc")})torch.manual_seed(1)flat = MGLPDSNet(**P_FLAT).to(DEVICE).eval()torch.manual_seed(1)mg = MGLPDSNet(**P_MG).to(DEVICE).eval()nparams = lambda m: sum(p.numel() for p in m.parameters())print(f"\nLPDSNet    levels={flat.levels}  pad_stride={flat.pad_stride}  "      f"params={nparams(flat):,}")print(f"MGLPDSNet  levels={mg.levels}  pad_stride={mg.pad_stride}  "      f"params={nparams(mg):,}")print(f"\n-> sizes divisible by {flat.pad_stride} are safe for LPDSNet, "      f"by {mg.pad_stride} for MGLPDSNet")

## 4. The mechanism, before any scoresRun the preprocessing the models actually run, and look at the grid it hands back.If the hypothesis is right, the two models get *different-sized* surrogates at a sizethat is not a multiple of 8, and identical ones when it is.

In [ ]:
S = (min(GT_FULL.shape[-2:]) // 8) * 8          # largest multiple of 8 that fitsprint(f"{'size':>6}  {'model':<10}{'pad_stride':>11}{'y~ grid':>12}   padded?")for size in (S - 4, S):    prob = make_problem(size)    for name, net in (("LPDSNet", flat), ("MGLPDSNet", mg)):        y_t, E_p, _ = kspace_pre_process(prob["y"], prob["E"], net.pad_stride)        grid = tuple(y_t.shape[-2:])        padded = "YES  <-- operator resampled" if grid != prob["size"] else "no"        print(f"{size:>6}  {name:<10}{net.pad_stride:>11}{str(grid):>12}   {padded}")    print()

## 5. The sweepFive sizes around `S`: two divisible by 8, three not. All are even, so LPDSNet's stride-2padding is empty at every one of them.

In [ ]:
SIZES = [S - 8, S - 6, S - 4, S - 2, S]rows = []for size in SIZES:    prob = make_problem(size)    with torch.no_grad():        x_flat, _ = flat(prob["y"], E=prob["E"], sigma=prob["sigma"])        x_mg, _ = mg(prob["y"], E=prob["E"], sigma=prob["sigma"])    x_adj = prob["E"].adjoint(prob["y"])    rows.append(dict(size=size, mod8=size % 8,                     zf=psnr(x_adj, prob["gt"]),                     flat=psnr(x_flat, prob["gt"]),                     mg=psnr(x_mg, prob["gt"]),                     flat_nrmse=nrmse(x_flat, prob["gt"]),                     mg_nrmse=nrmse(x_mg, prob["gt"])))hdr = f"{'size':>6}{'%8':>5}{'zero-fill':>11}{'LPDSNet':>10}{'MGLPDSNet':>11}{'gap dB':>9}"print(hdr); print("-" * len(hdr))for r in rows:    flag = "" if r["mod8"] == 0 else "   <-- MG pads"    print(f"{r['size']:>6}{r['mod8']:>5}{r['zf']:>11.2f}{r['flat']:>10.2f}"          f"{r['mg']:>11.2f}{r['mg'] - r['flat']:>9.2f}{flag}")div = [r for r in rows if r["mod8"] == 0]non = [r for r in rows if r["mod8"] != 0]print(f"\nmean gap (MG - LPDS), size % 8 == 0 : {np.mean([r['mg']-r['flat'] for r in div]):+.2f} dB")print(f"mean gap (MG - LPDS), size % 8 != 0 : {np.mean([r['mg']-r['flat'] for r in non]):+.2f} dB")print(f"LPDSNet spread across all sizes     : "      f"{max(r['flat'] for r in rows) - min(r['flat'] for r in rows):.2f} dB")

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))sz = [r["size"] for r in rows]ax.plot(sz, [r["flat"] for r in rows], "o-", label=f"LPDSNet (pad_stride {flat.pad_stride})")ax.plot(sz, [r["mg"] for r in rows], "s-", label=f"MGLPDSNet (pad_stride {mg.pad_stride})")ax.plot(sz, [r["zf"] for r in rows], "k:", lw=1, label="zero-filled $E^H y$")for r in rows:    if r["mod8"] == 0:        ax.axvline(r["size"], color="0.85", lw=8, zorder=0)ax.set_xlabel("image size (grey bands: divisible by 8)")ax.set_ylabel("PSNR (dB)")ax.set_title(f"At initialisation, {SOURCE} data, R={R}, $\\sigma$={SIGMA}")ax.set_xticks(sz); ax.legend(); ax.grid(alpha=.3)plt.tight_layout(); plt.show()

## 6. What it looks likeSame networks, same weights, two sizes four pixels apart.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 6.8))for row, size in enumerate((S, S - 4)):    prob = make_problem(size)    with torch.no_grad():        xf, _ = flat(prob["y"], E=prob["E"], sigma=prob["sigma"])        xm, _ = mg(prob["y"], E=prob["E"], sigma=prob["sigma"])    xa = prob["E"].adjoint(prob["y"])    vmax = float(prob["gt"].abs().max())    panels = [("ground truth", prob["gt"], None),              ("zero-filled", xa, psnr(xa, prob["gt"])),              ("LPDSNet", xf, psnr(xf, prob["gt"])),              ("MGLPDSNet", xm, psnr(xm, prob["gt"]))]    for ax, (title, img, p) in zip(axes[row], panels):        ax.imshow(img[0, 0].abs().cpu().numpy(), cmap="gray", vmin=0, vmax=vmax)        lab = title if p is None else f"{title}  {p:.2f} dB"        ax.set_title(lab, fontsize=10)        ax.axis("off")    tag = "divisible by 8" if size % 8 == 0 else "NOT divisible by 8"    axes[row][0].set_ylabel(f"{size}", fontsize=11)    axes[row][0].axis("on"); axes[row][0].set_xticks([]); axes[row][0].set_yticks([])    axes[row][0].set_title(f"ground truth\n{size} -- {tag}", fontsize=10)plt.tight_layout(); plt.show()

## 7. Control: is the V-cycle itself to blame?`alpha_x` and `alpha_z` scale the coarse-grid correction. At `alpha0 = 0` the correction isan exact no-op, so a V-cycle reduces to its fine-grid smoothing sweeps -- and`K = [6, [4, 4, 6]]` has `6 x 4 = 24` of them.If the multigrid machinery were responsible for the gap, killing it would close the gap.If instead `MGLPDSNet(alpha0=0)` lands **exactly** on `LPDSNet(K=24)` at a divisible sizeand still collapses at a non-divisible one, the fine path is shared and correct and thecoarse path is not the problem.

In [ ]:
torch.manual_seed(1)mg0 = MGLPDSNet(**dict(P_MG, alpha0=0.0)).to(DEVICE).eval()torch.manual_seed(1)flat24 = MGLPDSNet(**dict(P_FLAT, K=24)).to(DEVICE).eval()print(f"{'size':>6}{'%8':>5}{'MG alpha0=0':>14}{'LPDSNet K=24':>15}{'identical?':>12}")for size in (S, S - 4):    prob = make_problem(size)    with torch.no_grad():        a, _ = mg0(prob["y"], E=prob["E"], sigma=prob["sigma"])        b, _ = flat24(prob["y"], E=prob["E"], sigma=prob["sigma"])    same = "yes" if torch.allclose(a, b, atol=1e-5, rtol=1e-4) else "NO"    print(f"{size:>6}{size % 8:>5}{psnr(a, prob['gt']):>14.2f}"          f"{psnr(b, prob['gt']):>15.2f}{same:>12}")print("\n(the two collapse onto each other only where MGLPDSNet does not pad;")print(" at a non-divisible size they solve different problems, so they cannot match)")

## 8. Reading the result**Hypothesis confirmed** if: LPDSNet is flat across all five sizes, MGLPDSNet matches orbeats it at the two divisible sizes and drops sharply at the three others, and the`alpha0=0` control lands on `LPDSNet(K=24)` at the divisible size.**Hypothesis refuted** if: MGLPDSNet is uniformly worse at every size including thedivisible ones, or both models degrade together. Then the padding is incidental and theV-cycle is the thing to look at -- start with `alpha0` (the configs use `1.0`, while`models/multigrid.py::VCycle` defaults to `1e-1`) and with the FAS corrections in`PDObjectiveDownsample`.### If confirmed, the fix1. Set `crop_size` (or `center_crop`) to a multiple of 8 in   `scripts/make_mg_recon_configs.py`. The configs currently use `crop_size: null`, i.e.   full FOV, and fastMRI brain matrix sizes vary volume to volume -- so this fires on some   slices and not others, which reads as instability rather than as a bug.2. Make the k-space path loud. `preproc="identity"` already calls `_check_grid` and raises;   `preproc="kspace"` pads silently. A warn-once naming the model and size would have made   this immediate.3. Re-check the other pair before trusting any grid result: `altsplit`'s denoiser is   `K=6` (1 level, `pad_stride` 2) and `mgaltsplit`'s is `K=[1, [4, 4, 6]]` (3 levels,   `pad_stride` 8), and `LPDS_DENOISER` strips `preproc` so both inherit the   `"kspace"` default. The same asymmetry is available there.Until sizes are a multiple of 8 for every model in the grid, flat-vs-multigrid comparisonsare confounded: the two arms are not solving the same problem.